# 可信井 log-AI 数十米尺度分解实验

本 notebook 是前三轮井曲线残差实验的第四轮。它先实测六口可信井位置的真实地震同号波瓣宽度，再把既有三带高斯分解从几米尺度移动到 10–75 m，并复用同一深度域正演检查。

本轮只审计尺度和形态，不训练网络，也不把过零点或局部峰解释为地质边界。


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display
from scipy import signal
from scipy.ndimage import gaussian_filter1d

repo_root = Path.cwd().resolve()
if not (repo_root / "src").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "src").is_dir():
    raise RuntimeError("Could not locate repository root containing src/.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.config.workflow import WorkflowConfig
from cup.physics.numpy_backend import forward_depth, velocity_from_ai
from cup.seismic.survey import open_survey, segy_options_from_config
from cup.synthetic.core.signal import finite_support_fir, valid_filter_decimate
from cup.utils.io import resolve_relative_path

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180, "axes.grid": True, "grid.alpha": 0.20, "font.size": 8.5})
print(f"Repository: {repo_root}")

In [ ]:
# 实验旋钮集中在本 cell。BODY_SCALE_SMOKE=1 时只跑 NW11 和两组候选。
SMOKE = os.environ.get("BODY_SCALE_SMOKE", "0") == "1"
SOURCE_RUN_DIR = repo_root / "experiments" / "well_residual_decomposition" / "results" / "20260810_gaussian_scale_space"
COMMON_CONFIG = repo_root / "experiments" / "common" / "common.yaml"
RUN_ID = "20260810_body_scale_decomposition_smoke" if SMOKE else "20260810_body_scale_decomposition"
OUTPUT_DIR = repo_root / "experiments" / "well_residual_decomposition" / "results" / RUN_ID

TRUSTED_WELLS = ("2-ANP-2A-RJS", "L1-NW1", "L5-NW5", "L9-NW4A", "NW11", "NW8")
CANDIDATES = {
    "F10_B50": {"fine_fwhm_m": 10.0, "broad_fwhm_m": 50.0},
    "F15_B50": {"fine_fwhm_m": 15.0, "broad_fwhm_m": 50.0},
    "F10_B75": {"fine_fwhm_m": 10.0, "broad_fwhm_m": 75.0},
    "F15_B75": {"fine_fwhm_m": 15.0, "broad_fwhm_m": 75.0},
}
REFERENCE_CANDIDATE = "F15_B50"
EVENT_THRESHOLD_FRACTIONS = (0.05, 0.10, 0.20)
REFERENCE_EVENT_THRESHOLD = 0.10
MAX_REVIEW_EVENTS_PER_WELL = 3
MODEL_GRID_INTERVAL_M = 5.0
FORWARD_OUTPUT_CHUNK_SIZE = 32
MIN_RUN_BROAD_FWHM_MULTIPLE = 1.0

if SMOKE:
    TRUSTED_WELLS = ("NW11",)
    CANDIDATES = {key: CANDIDATES[key] for key in ("F10_B50", "F15_B75")}
    REFERENCE_CANDIDATE = "F10_B50"
    MAX_REVIEW_EVENTS_PER_WELL = 1

for required in (SOURCE_RUN_DIR / "manifest.json", SOURCE_RUN_DIR / "metrics.csv", COMMON_CONFIG):
    if not required.exists():
        raise FileNotFoundError(required)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)
(OUTPUT_DIR / "wells").mkdir(exist_ok=True)
print(f"Smoke: {SMOKE}")
print(f"Output: {OUTPUT_DIR}")

## 1. 加载冻结井曲线、子波和井位真实地震

井曲线、层位和井上合成地震读取第一轮已发布 artifact。真实地震直接从当前深度域 survey 按井口 XY 读取，不依赖旧 GINN 预测产物。


In [ ]:
def resolve_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else repo_root / path


def finite_runs(mask):
    mask = np.asarray(mask, dtype=bool)
    padded = np.concatenate(([False], mask, [False]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    return tuple(slice(int(start), int(stop)) for start, stop in changes.reshape(-1, 2))


def rms(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(np.sqrt(np.mean(values**2))) if values.size else np.nan


def safe_corr(left, right, support):
    support = np.asarray(support, dtype=bool) & np.isfinite(left) & np.isfinite(right)
    if np.count_nonzero(support) < 3:
        return np.nan
    a = np.asarray(left, dtype=np.float64)[support]
    b = np.asarray(right, dtype=np.float64)[support]
    if np.std(a) <= 1e-12 or np.std(b) <= 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def wavelet_frequency_summary(time_s, amplitude):
    time_s = np.asarray(time_s, dtype=np.float64)
    amplitude = np.asarray(amplitude, dtype=np.float64)
    dt_s = float(np.median(np.diff(time_s)))
    centered = amplitude - np.mean(amplitude)
    frequency = np.fft.rfftfreq(centered.size, d=dt_s)
    power = np.abs(np.fft.rfft(centered)) ** 2
    power[0] = 0.0
    probability = power / np.sum(power)
    cumulative = np.cumsum(probability)
    quantile = lambda q: float(frequency[np.searchsorted(cumulative, q)])
    return {
        "peak_hz": float(frequency[int(np.argmax(power))]),
        "energy_centroid_hz": float(np.sum(frequency * probability)),
        "energy_median_hz": quantile(0.50),
    }


source_manifest = json.loads((SOURCE_RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
source_metrics = pd.read_csv(SOURCE_RUN_DIR / "metrics.csv")
source_well_metrics = source_metrics.sort_values(["well_name", "setting"]).groupby("well_name", as_index=True).first()
batch_metrics = pd.read_csv(resolve_path(source_manifest["inputs"]["wavelet_batch_metrics"])).set_index(
    "well_name", drop=False
)
forward_inputs = json.loads(resolve_path(source_manifest["inputs"]["forward_inputs"]).read_text(encoding="utf-8"))
wavelet_frame = pd.read_csv(resolve_path(forward_inputs["wavelet"]["path"]))
wavelet_time_s = wavelet_frame["time_s"].to_numpy(dtype=np.float64)
wavelet_amplitude = wavelet_frame["amplitude"].to_numpy(dtype=np.float64)
wavelet_summary = wavelet_frequency_summary(wavelet_time_s, wavelet_amplitude)
relation = forward_inputs["ai_velocity_relation"]
AI_VP_A = float(relation["a"])
AI_VP_B = float(relation["b"])

common_mapping = yaml.safe_load(COMMON_CONFIG.read_text(encoding="utf-8"))
workflow = WorkflowConfig.from_mapping(common_mapping)
if workflow.seismic.domain != "depth" or workflow.seismic.depth_basis != "tvdss":
    raise ValueError("This experiment requires depth/TVDSS seismic.")
data_root = resolve_relative_path(workflow.data_root, root=repo_root)
seismic_path = resolve_relative_path(workflow.seismic.file, root=data_root)
seismic_options = segy_options_from_config(workflow.seismic.as_dict()) or None
survey = open_survey(seismic_path, seismic_type=workflow.seismic.type, segy_options=seismic_options)

wells = {}
tuning_rows = []
for well_name in TRUSTED_WELLS:
    artifact_path = resolve_path(source_manifest["well_artifacts"][well_name])
    with np.load(artifact_path, allow_pickle=False) as artifact:
        depth = artifact["tvdss_m"].astype(np.float64)
        log_ai = artifact["well_log_ai"].astype(np.float64)
        valid = artifact["well_valid"].astype(bool)
        horizon_names = artifact["horizon_names"].astype(str)
        horizon_depths = artifact["horizon_tvdss_m"].astype(np.float64)
        synthetic_full = artifact["g100_synthetic_full"].astype(np.float64)
    intervals = np.diff(depth)
    dz_m = float(np.median(intervals))
    if np.any(intervals <= 0.0) or not np.allclose(intervals, dz_m, rtol=1e-5, atol=1e-6):
        raise ValueError(f"{well_name}: source well axis is not regular.")
    row = batch_metrics.loc[well_name]
    real_trace = survey.read_trace_at_xy(float(row["well_x"]), float(row["well_y"]), domain="depth")
    real_depth = np.asarray(real_trace.basis, dtype=np.float64)
    real_seismic = np.asarray(real_trace.values, dtype=np.float64)
    horizons = dict(zip(horizon_names.tolist(), horizon_depths.tolist()))
    median_vp_mps = float(row["median_vp_mps"])
    for frequency_name, frequency_hz in wavelet_summary.items():
        tuning_rows.append(
            {
                "well_name": well_name,
                "frequency_definition": frequency_name,
                "frequency_hz": frequency_hz,
                "median_vp_mps": median_vp_mps,
                "tuning_scale_m": median_vp_mps / (4.0 * frequency_hz),
            }
        )
    wells[well_name] = {
        "well_name": well_name,
        "tvdss_m": depth,
        "log_ai": log_ai,
        "valid": valid,
        "dz_m": dz_m,
        "horizons": horizons,
        "synthetic_full": synthetic_full,
        "real_depth_m": real_depth,
        "real_seismic": real_seismic,
        "median_vp_mps": median_vp_mps,
        "tie_corr": float(row["corr"]),
        "source_artifact": str(artifact_path),
        "well_x": float(row["well_x"]),
        "well_y": float(row["well_y"]),
    }
del survey
tuning_scales = pd.DataFrame(tuning_rows)
display(pd.DataFrame([wavelet_summary]))
display(tuning_scales)

## 2. L0：真实与井上合成地震事件宽度

事件定义为目标层段内连续同号波瓣。每条道先减去目标层段中位数，再按本道绝对振幅 P95 的 5% / 10% / 20% 过滤弱波瓣。三档阈值用于敏感性分析；10% 只作为图件参考。


In [ ]:
def axis_runs(axis, valid):
    axis = np.asarray(axis, dtype=np.float64)
    valid = np.asarray(valid, dtype=bool)
    positive_steps = np.diff(axis)
    nominal = float(np.median(positive_steps[positive_steps > 0.0]))
    runs = []
    for run in finite_runs(valid):
        local_breaks = np.flatnonzero(np.diff(axis[run]) > 1.5 * nominal) + run.start + 1
        edges = np.concatenate(([run.start], local_breaks, [run.stop]))
        runs.extend(slice(int(start), int(stop)) for start, stop in zip(edges[:-1], edges[1:]) if stop > start)
    return tuple(runs), nominal


def same_polarity_lobes(axis, amplitude, valid, *, well_name, source, threshold_fraction):
    axis = np.asarray(axis, dtype=np.float64)
    amplitude = np.asarray(amplitude, dtype=np.float64)
    valid = np.asarray(valid, dtype=bool) & np.isfinite(axis) & np.isfinite(amplitude)
    if np.count_nonzero(valid) < 3:
        return []
    centered = amplitude - float(np.median(amplitude[valid]))
    amplitude_scale = float(np.percentile(np.abs(centered[valid]), 95.0))
    if not np.isfinite(amplitude_scale) or amplitude_scale <= 0.0:
        return []
    runs, nominal_step = axis_runs(axis, valid)
    rows = []
    event_index = 0
    for run in runs:
        local = centered[run]
        local_axis = axis[run]
        if local.size < 1:
            continue
        sign_positive = local >= 0.0
        changes = np.flatnonzero(sign_positive[1:] != sign_positive[:-1]) + 1
        edges = np.concatenate(([0], changes, [local.size]))
        for start, stop in zip(edges[:-1], edges[1:]):
            values = local[start:stop]
            if values.size == 0:
                continue
            peak = float(np.max(np.abs(values)))
            if peak < float(threshold_fraction) * amplitude_scale:
                continue
            top = float(local_axis[start] - 0.5 * nominal_step)
            bottom = float(local_axis[stop - 1] + 0.5 * nominal_step)
            area = (
                float(np.trapezoid(np.abs(values), local_axis[start:stop])) if values.size >= 2 else peak * nominal_step
            )
            rows.append(
                {
                    "well_name": well_name,
                    "source": source,
                    "threshold_fraction": float(threshold_fraction),
                    "event_index": event_index,
                    "top_m": top,
                    "bottom_m": bottom,
                    "width_m": bottom - top,
                    "polarity": 1 if sign_positive[start] else -1,
                    "peak_abs": peak,
                    "area_abs": area,
                    "trace_abs_p95": amplitude_scale,
                    "baseline_removed": float(np.median(amplitude[valid])),
                }
            )
            event_index += 1
    return rows


event_rows = []
for well_name, well in wells.items():
    horizon_values = np.asarray(list(well["horizons"].values()), dtype=np.float64)
    target_top, target_bottom = float(np.min(horizon_values)), float(np.max(horizon_values))
    real_valid = (
        np.isfinite(well["real_seismic"])
        & (well["real_depth_m"] >= target_top)
        & (well["real_depth_m"] <= target_bottom)
    )
    synthetic_valid = (
        np.isfinite(well["synthetic_full"]) & (well["tvdss_m"] >= target_top) & (well["tvdss_m"] <= target_bottom)
    )
    synthetic_indices = np.flatnonzero(synthetic_valid)
    for threshold in EVENT_THRESHOLD_FRACTIONS:
        event_rows.extend(
            same_polarity_lobes(
                well["real_depth_m"],
                well["real_seismic"],
                real_valid,
                well_name=well_name,
                source="real_seismic",
                threshold_fraction=threshold,
            )
        )
        if synthetic_indices.size >= 3:
            event_rows.extend(
                same_polarity_lobes(
                    well["tvdss_m"][synthetic_indices],
                    well["synthetic_full"][synthetic_indices],
                    np.ones(synthetic_indices.size, dtype=bool),
                    well_name=well_name,
                    source="well_synthetic",
                    threshold_fraction=threshold,
                )
            )

event_scale = pd.DataFrame(event_rows)
if event_scale.empty:
    raise ValueError("No seismic lobes survived the event-scale audit.")
event_scale_path = OUTPUT_DIR / "event_scale.csv"
event_scale.to_csv(event_scale_path, index=False)
event_sensitivity = (
    event_scale.groupby(["well_name", "source", "threshold_fraction"])["width_m"]
    .agg(["count", "median", lambda s: s.quantile(0.10), lambda s: s.quantile(0.90)])
    .reset_index()
)
event_sensitivity.columns = [
    "well_name",
    "source",
    "threshold_fraction",
    "event_count",
    "width_p50_m",
    "width_p10_m",
    "width_p90_m",
]
event_sensitivity.to_csv(OUTPUT_DIR / "sensitivity.csv", index=False)
tuning_scales.to_csv(OUTPUT_DIR / "tuning_scale.csv", index=False)
display(event_sensitivity)

## 3. L1/L2：数十米三带分解与正演贡献

四组候选只改变 fine/broad FWHM。全部曲线在 0.1 m 井轴上严格重建；正演前使用现有有限支撑投影降到 5 m 模型网格。


In [ ]:
def smooth_valid_runs(values, valid, dz_m, fwhm_m, minimum_run_m):
    sigma_samples = (float(fwhm_m) / 2.354820045) / float(dz_m)
    minimum_samples = max(3, int(np.ceil(float(minimum_run_m) / float(dz_m))))
    smoothed = np.full(values.shape, np.nan, dtype=np.float64)
    support = np.zeros(values.shape, dtype=bool)
    for run in finite_runs(valid & np.isfinite(values)):
        if run.stop - run.start < minimum_samples:
            continue
        smoothed[run] = gaussian_filter1d(values[run], sigma=sigma_samples, mode="reflect", truncate=4.0)
        support[run] = True
    return smoothed, support


def three_band_decomposition(well, fine_fwhm_m, broad_fwhm_m):
    if not (0.0 < fine_fwhm_m < broad_fwhm_m):
        raise ValueError("Expected 0 < fine FWHM < broad FWHM.")
    minimum_run_m = MIN_RUN_BROAD_FWHM_MULTIPLE * broad_fwhm_m
    fine, fine_support = smooth_valid_runs(well["log_ai"], well["valid"], well["dz_m"], fine_fwhm_m, minimum_run_m)
    broad, broad_support = smooth_valid_runs(well["log_ai"], well["valid"], well["dz_m"], broad_fwhm_m, minimum_run_m)
    support = fine_support & broad_support
    detail = np.where(support, fine - broad, np.nan)
    ultrafine = np.where(support, well["log_ai"] - fine, np.nan)
    reconstructed = np.where(support, broad + detail + ultrafine, np.nan)
    if not np.any(support):
        raise ValueError(f"{well['well_name']}: no support for three-band decomposition.")
    parity = float(np.max(np.abs(reconstructed[support] - well["log_ai"][support])))
    if parity > 1e-10:
        raise ValueError(f"Three-band reconstruction parity failed: {parity:.6g}")
    return {
        "fine_fwhm_m": float(fine_fwhm_m),
        "broad_fwhm_m": float(broad_fwhm_m),
        "support": support,
        "broad_log_ai": broad,
        "body_smooth_log_ai": fine,
        "body_scale_detail": detail,
        "ultrafine": ultrafine,
        "reconstruction_max_abs": parity,
    }


def projected_forward_log_ai(depth_m, log_ai, support, highres_interval_m):
    depth_m = np.asarray(depth_m, dtype=np.float64)
    log_ai = np.asarray(log_ai, dtype=np.float64)
    support = np.asarray(support, dtype=bool)
    output = np.full(log_ai.shape, np.nan, dtype=np.float64)
    factor_float = MODEL_GRID_INTERVAL_M / float(highres_interval_m)
    factor = int(round(factor_float))
    if factor < 1 or not np.isclose(factor_float, factor, rtol=0.0, atol=1e-6):
        raise ValueError("High-resolution LAS axis is not nested with the 5 m model interval.")
    taps = finite_support_fir(factor)
    for run in finite_runs(support & np.isfinite(log_ai)):
        if run.stop - run.start < 3:
            continue
        model_log_ai, model_support = valid_filter_decimate(log_ai[run], factor=factor, taps=taps)
        model_depth = depth_m[run][::factor]
        model_indices = np.arange(run.start, run.stop, factor, dtype=np.int64)
        if model_log_ai.shape != model_depth.shape or model_depth.shape != model_indices.shape:
            raise ValueError("Projected model arrays have inconsistent shapes.")
        if model_log_ai.size < 2 or not np.any(model_support):
            continue
        local_ai = np.exp(model_log_ai)
        local_vp = velocity_from_ai(local_ai, a=AI_VP_A, b=AI_VP_B)
        synthetic = forward_depth(
            model_log_ai,
            local_vp,
            model_depth,
            wavelet_time_s,
            wavelet_amplitude,
            output_chunk_size=FORWARD_OUTPUT_CHUNK_SIZE,
        )
        output[model_indices[model_support]] = synthetic[model_support]
    return output


def zero_crossing_intervals(values, support, dz_m):
    intervals = []
    for run in finite_runs(np.asarray(support, dtype=bool) & np.isfinite(values)):
        local = np.asarray(values[run], dtype=np.float64)
        if local.size < 2:
            continue
        changes = np.flatnonzero((local[1:] >= 0.0) != (local[:-1] >= 0.0)) + 1
        edges = np.concatenate(([0], changes, [local.size]))
        intervals.extend((np.diff(edges) * float(dz_m)).tolist())
    return np.asarray(intervals, dtype=np.float64)


def dominant_wavelength(values, support, dz_m):
    runs = finite_runs(np.asarray(support, dtype=bool) & np.isfinite(values))
    if not runs:
        return np.nan
    run = max(runs, key=lambda item: item.stop - item.start)
    local = signal.detrend(np.asarray(values[run], dtype=np.float64))
    if local.size < 16:
        return np.nan
    power = np.abs(np.fft.rfft(local * np.hanning(local.size))) ** 2
    frequency = np.fft.rfftfreq(local.size, d=float(dz_m))
    valid_frequency = frequency > 0.0
    if not np.any(valid_frequency):
        return np.nan
    selected = np.flatnonzero(valid_frequency)[int(np.argmax(power[valid_frequency]))]
    return float(1.0 / frequency[selected])


all_results = {}
metric_rows = []
artifact_paths = {}
for well_name, well in wells.items():
    target_values = np.asarray(list(well["horizons"].values()), dtype=np.float64)
    target = (well["tvdss_m"] >= np.min(target_values)) & (well["tvdss_m"] <= np.max(target_values))
    candidate_results = {}
    for candidate_name, config in CANDIDATES.items():
        result = three_band_decomposition(well, config["fine_fwhm_m"], config["broad_fwhm_m"])
        synthetic_body = projected_forward_log_ai(
            well["tvdss_m"], result["body_smooth_log_ai"], result["support"], well["dz_m"]
        )
        synthetic_broad = projected_forward_log_ai(
            well["tvdss_m"], result["broad_log_ai"], result["support"], well["dz_m"]
        )
        result["synthetic_body"] = synthetic_body
        result["synthetic_broad"] = synthetic_broad
        result["synthetic_detail_contribution"] = synthetic_body - synthetic_broad
        result["synthetic_ultrafine_contribution"] = well["synthetic_full"] - synthetic_body
        common = (
            target
            & result["support"]
            & np.isfinite(well["synthetic_full"])
            & np.isfinite(synthetic_body)
            & np.isfinite(synthetic_broad)
        )
        full_rms = rms(well["synthetic_full"][common])
        intervals = zero_crossing_intervals(result["body_scale_detail"], result["support"] & target, well["dz_m"])
        metric_rows.append(
            {
                "well_name": well_name,
                "candidate": candidate_name,
                "fine_fwhm_m": result["fine_fwhm_m"],
                "broad_fwhm_m": result["broad_fwhm_m"],
                "fine_tuning_fraction_energy_median": result["fine_fwhm_m"]
                / float(source_well_metrics.loc[well_name, "tuning_scale_m"]),
                "broad_tuning_fraction_energy_median": result["broad_fwhm_m"]
                / float(source_well_metrics.loc[well_name, "tuning_scale_m"]),
                "supported_samples": int(np.count_nonzero(result["support"] & target)),
                "reconstruction_max_abs": result["reconstruction_max_abs"],
                "body_scale_detail_rms": rms(result["body_scale_detail"][result["support"] & target]),
                "ultrafine_rms": rms(result["ultrafine"][result["support"] & target]),
                "detail_zero_crossing_p50_m": float(np.median(intervals)) if intervals.size else np.nan,
                "detail_zero_crossing_p90_m": float(np.quantile(intervals, 0.90)) if intervals.size else np.nan,
                "detail_dominant_wavelength_m": dominant_wavelength(
                    result["body_scale_detail"], result["support"] & target, well["dz_m"]
                ),
                "forward_status": "ok"
                if np.count_nonzero(common) >= 3 and np.isfinite(full_rms) and full_rms > 0.0
                else "unavailable_no_common_support",
                "broad_forward_corr": safe_corr(well["synthetic_full"], synthetic_broad, common),
                "body_reconstruction_forward_corr": safe_corr(well["synthetic_full"], synthetic_body, common),
                "detail_forward_rms_ratio": rms(result["synthetic_detail_contribution"][common]) / full_rms
                if np.isfinite(full_rms) and full_rms > 0.0
                else np.nan,
                "ultrafine_forward_rms_ratio": rms(result["synthetic_ultrafine_contribution"][common]) / full_rms
                if np.isfinite(full_rms) and full_rms > 0.0
                else np.nan,
            }
        )
        candidate_results[candidate_name] = result
    all_results[well_name] = candidate_results
    artifact_path = OUTPUT_DIR / "wells" / f"{well_name}.npz"
    payload = {
        "tvdss_m": well["tvdss_m"],
        "well_log_ai": well["log_ai"],
        "well_valid": well["valid"],
        "real_depth_m": well["real_depth_m"],
        "real_seismic": well["real_seismic"],
        "synthetic_full": well["synthetic_full"],
        "horizon_names": np.asarray(list(well["horizons"].keys()), dtype="U64"),
        "horizon_tvdss_m": np.asarray(list(well["horizons"].values()), dtype=np.float64),
    }
    for candidate_name, result in candidate_results.items():
        prefix = candidate_name.lower()
        for key in (
            "support",
            "broad_log_ai",
            "body_smooth_log_ai",
            "body_scale_detail",
            "ultrafine",
            "synthetic_body",
            "synthetic_broad",
            "synthetic_detail_contribution",
            "synthetic_ultrafine_contribution",
        ):
            payload[f"{prefix}_{key}"] = result[key]
    np.savez_compressed(artifact_path, **payload)
    artifact_paths[well_name] = str(artifact_path)
    print(f"completed decomposition: {well_name}")

candidate_metrics = pd.DataFrame(metric_rows).sort_values(["well_name", "candidate"])
candidate_metrics_path = OUTPUT_DIR / "candidate_metrics.csv"
candidate_metrics.to_csv(candidate_metrics_path, index=False)
display(candidate_metrics)

## 4. 固定事件窗口图件

每口井使用真实地震中 10% P95 门槛下最强的三个同号波瓣。图件中的 detail 共用横轴，避免缩放制造更平滑或更强的视觉印象。


In [ ]:
CANDIDATE_COLORS = {"F10_B50": "#1f77b4", "F15_B50": "#ff7f0e", "F10_B75": "#2ca02c", "F15_B75": "#d62728"}


def target_limits(well):
    values = np.asarray(list(well["horizons"].values()), dtype=np.float64)
    return float(np.min(values)), float(np.max(values))


def selected_events(well_name):
    well = wells[well_name]
    reference = all_results[well_name][REFERENCE_CANDIDATE]
    selected = event_scale[
        (event_scale["well_name"] == well_name)
        & (event_scale["source"] == "real_seismic")
        & np.isclose(event_scale["threshold_fraction"], REFERENCE_EVENT_THRESHOLD)
    ].copy()
    keep = []
    for _, event in selected.iterrows():
        highres_window = (well["tvdss_m"] >= event["top_m"]) & (well["tvdss_m"] <= event["bottom_m"])
        highres_count = int(np.count_nonzero(highres_window))
        decomposition_fraction = float(np.count_nonzero(highres_window & reference["support"])) / max(highres_count, 1)
        forward_support = (
            highres_window & np.isfinite(well["synthetic_full"]) & np.isfinite(reference["synthetic_body"])
        )
        expected_model_samples = max(3, int(round(float(event["width_m"]) / MODEL_GRID_INTERVAL_M)))
        keep.append(
            decomposition_fraction >= 0.80
            and np.count_nonzero(forward_support) >= max(3, int(0.60 * expected_model_samples))
        )
    selected = selected.loc[np.asarray(keep, dtype=bool)]
    selected = selected.sort_values("peak_abs", ascending=False).head(MAX_REVIEW_EVENTS_PER_WELL).sort_values("top_m")
    return selected.to_dict("records")


def normalized(values, support):
    output = np.full(np.asarray(values).shape, np.nan, dtype=np.float64)
    support = np.asarray(support, dtype=bool) & np.isfinite(values)
    if np.any(support):
        scale = float(np.percentile(np.abs(np.asarray(values)[support]), 95.0))
        if scale > 0.0:
            output[support] = np.asarray(values)[support] / scale
    return output


def normalized_xy(axis, values, support):
    axis = np.asarray(axis, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    support = np.asarray(support, dtype=bool) & np.isfinite(axis) & np.isfinite(values)
    indices = np.flatnonzero(support)
    if indices.size == 0:
        return np.empty(0, dtype=np.float64), np.empty(0, dtype=np.float64)
    scale = float(np.percentile(np.abs(values[indices]), 95.0))
    if not np.isfinite(scale) or scale <= 0.0:
        return np.zeros(indices.size, dtype=np.float64), axis[indices]
    return values[indices] / scale, axis[indices]


def shade_events(axis, panel, rows):
    for row in rows:
        panel.axhspan(
            row["top_m"], row["bottom_m"], color=("#d62728" if row["polarity"] > 0 else "#1f77b4"), alpha=0.12
        )


event_candidate_rows = []
figure_manifest = {}
for well_name, well in wells.items():
    events = selected_events(well_name)
    if not events:
        print(f"warning: no reference events for {well_name}")
        continue
    well_dir = OUTPUT_DIR / "figures" / well_name
    well_dir.mkdir(parents=True, exist_ok=True)
    panel_count = len(CANDIDATES) + 3
    forward_column = panel_count - 1
    fig, axes = plt.subplots(
        len(events),
        panel_count,
        figsize=(2.45 * panel_count, 3.4 * len(events)),
        constrained_layout=True,
        squeeze=False,
    )
    for event_row, event in enumerate(events):
        width = float(event["width_m"])
        context = max(20.0, 0.25 * width)
        lower, upper = event["top_m"] - context, event["bottom_m"] + context
        real_view = (
            (well["real_depth_m"] >= lower) & (well["real_depth_m"] <= upper) & np.isfinite(well["real_seismic"])
        )
        log_view = (well["tvdss_m"] >= lower) & (well["tvdss_m"] <= upper) & well["valid"]
        real_x, real_y = normalized_xy(well["real_depth_m"], well["real_seismic"], real_view)
        axes[event_row, 0].plot(real_x, real_y, color="black", linewidth=1.0)
        axes[event_row, 0].axhspan(event["top_m"], event["bottom_m"], color="#9467bd", alpha=0.12)
        reference = all_results[well_name][REFERENCE_CANDIDATE]
        axes[event_row, 1].plot(well["log_ai"], well["tvdss_m"], color="0.25", linewidth=0.7, label="full")
        axes[event_row, 1].plot(
            reference["broad_log_ai"], well["tvdss_m"], color="#ff7f0e", linewidth=1.2, label="broad"
        )
        axes[event_row, 1].plot(
            reference["body_smooth_log_ai"], well["tvdss_m"], color="#2ca02c", linewidth=1.0, label="broad+detail"
        )
        detail_values = np.concatenate(
            [result["body_scale_detail"][log_view & result["support"]] for result in all_results[well_name].values()]
        )
        detail_limit = max(float(np.percentile(np.abs(detail_values), 99.0)), 1e-6)
        for column, (candidate_name, result) in enumerate(all_results[well_name].items(), start=2):
            axes[event_row, column].plot(
                result["body_scale_detail"], well["tvdss_m"], color=CANDIDATE_COLORS[candidate_name], linewidth=1.0
            )
            axes[event_row, column].axvline(0.0, color="0.5", linewidth=0.5)
            axes[event_row, column].set_xlim(-detail_limit, detail_limit)
            local_lobes = same_polarity_lobes(
                well["tvdss_m"],
                result["body_scale_detail"],
                log_view & result["support"],
                well_name=well_name,
                source=candidate_name,
                threshold_fraction=0.15,
            )
            event_candidate_rows.append(
                {
                    "well_name": well_name,
                    "event_rank": event_row + 1,
                    "event_top_m": event["top_m"],
                    "event_bottom_m": event["bottom_m"],
                    "event_width_m": width,
                    "candidate": candidate_name,
                    "major_same_sign_interval_count": len(local_lobes),
                    "major_interval_width_p50_m": float(np.median([item["width_m"] for item in local_lobes]))
                    if local_lobes
                    else np.nan,
                }
            )
        synthetic_view = (
            (well["tvdss_m"] >= lower)
            & (well["tvdss_m"] <= upper)
            & np.isfinite(well["synthetic_full"])
            & np.isfinite(reference["synthetic_body"])
        )
        full_x, forward_y = normalized_xy(well["tvdss_m"], well["synthetic_full"], synthetic_view)
        retained_x, retained_y = normalized_xy(well["tvdss_m"], reference["synthetic_body"], synthetic_view)
        axes[event_row, forward_column].plot(full_x, forward_y, color="black", linewidth=1.0, label="full forward")
        axes[event_row, forward_column].plot(
            retained_x, retained_y, color="#2ca02c", linewidth=1.0, label="broad+detail"
        )
        for column in range(panel_count):
            axes[event_row, column].set_ylim(upper, lower)
        axes[event_row, 0].set_ylabel(f"event {event_row + 1}\nTVDSS (m)")
    titles = ["real seismic", "full / retained", *list(CANDIDATES), "forward check"]
    for column, title in enumerate(titles):
        axes[0, column].set_title(title)
    axes[0, 1].legend(fontsize=6)
    axes[0, forward_column].legend(fontsize=6)
    figure_path = well_dir / "body_scale_candidate_comparison.png"
    fig.suptitle(f"{well_name} | real-event windows | tens-of-metres decomposition")
    fig.savefig(figure_path, bbox_inches="tight")
    plt.close(fig)
    figure_manifest[well_name] = str(figure_path)

event_candidate_metrics = pd.DataFrame(event_candidate_rows)
event_candidate_metrics.to_csv(OUTPUT_DIR / "event_candidate_metrics.csv", index=False)
print(f"Wrote {len(figure_manifest)} per-well comparison figures.")

In [ ]:
# 图 1：每口井真实地震与井上合成地震的事件宽度。
fig, axes = plt.subplots(len(wells), 2, figsize=(8.5, 3.0 * len(wells)), constrained_layout=True, squeeze=False)
for row_index, (well_name, well) in enumerate(wells.items()):
    lower, upper = target_limits(well)
    reference_rows = event_scale[
        (event_scale["well_name"] == well_name)
        & np.isclose(event_scale["threshold_fraction"], REFERENCE_EVENT_THRESHOLD)
    ]
    real_rows = reference_rows[reference_rows["source"] == "real_seismic"].to_dict("records")
    synthetic_rows = reference_rows[reference_rows["source"] == "well_synthetic"].to_dict("records")
    real_view = (well["real_depth_m"] >= lower) & (well["real_depth_m"] <= upper) & np.isfinite(well["real_seismic"])
    synthetic_view = (well["tvdss_m"] >= lower) & (well["tvdss_m"] <= upper) & np.isfinite(well["synthetic_full"])
    real_x, real_y = normalized_xy(well["real_depth_m"], well["real_seismic"], real_view)
    synthetic_x, synthetic_y = normalized_xy(well["tvdss_m"], well["synthetic_full"], synthetic_view)
    axes[row_index, 0].plot(real_x, real_y, color="black", linewidth=0.9)
    axes[row_index, 1].plot(synthetic_x, synthetic_y, color="black", linewidth=0.9)
    shade_events(well["real_depth_m"], axes[row_index, 0], real_rows)
    shade_events(well["tvdss_m"], axes[row_index, 1], synthetic_rows)
    for axis in axes[row_index]:
        axis.set_ylim(upper, lower)
        axis.set_xlim(-1.2, 1.2)
    axes[row_index, 0].set_ylabel(f"{well_name}\nTVDSS (m)")
axes[0, 0].set_title("real seismic | accepted lobes")
axes[0, 1].set_title("well synthetic | accepted lobes")
event_atlas_path = OUTPUT_DIR / "figures" / "real_event_scale_atlas.png"
fig.savefig(event_atlas_path, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(event_atlas_path)))

# 图 2：每口井最强真实事件的候选对照。
panel_count = len(CANDIDATES) + 3
forward_column = panel_count - 1
fig, axes = plt.subplots(
    len(wells), panel_count, figsize=(2.45 * panel_count, 3.0 * len(wells)), constrained_layout=True, squeeze=False
)
for row_index, (well_name, well) in enumerate(wells.items()):
    events = selected_events(well_name)
    if not events:
        continue
    event = max(events, key=lambda item: item["peak_abs"])
    context = max(20.0, 0.25 * float(event["width_m"]))
    lower, upper = event["top_m"] - context, event["bottom_m"] + context
    real_view = (well["real_depth_m"] >= lower) & (well["real_depth_m"] <= upper) & np.isfinite(well["real_seismic"])
    log_view = (well["tvdss_m"] >= lower) & (well["tvdss_m"] <= upper) & well["valid"]
    real_x, real_y = normalized_xy(well["real_depth_m"], well["real_seismic"], real_view)
    axes[row_index, 0].plot(real_x, real_y, color="black", linewidth=1.0)
    reference = all_results[well_name][REFERENCE_CANDIDATE]
    axes[row_index, 1].plot(well["log_ai"], well["tvdss_m"], color="0.25", linewidth=0.7)
    axes[row_index, 1].plot(reference["broad_log_ai"], well["tvdss_m"], color="#ff7f0e", linewidth=1.1)
    axes[row_index, 1].plot(reference["body_smooth_log_ai"], well["tvdss_m"], color="#2ca02c", linewidth=1.0)
    detail_values = np.concatenate(
        [result["body_scale_detail"][log_view & result["support"]] for result in all_results[well_name].values()]
    )
    detail_limit = max(float(np.percentile(np.abs(detail_values), 99.0)), 1e-6)
    for column, (candidate_name, result) in enumerate(all_results[well_name].items(), start=2):
        axes[row_index, column].plot(
            result["body_scale_detail"], well["tvdss_m"], color=CANDIDATE_COLORS[candidate_name], linewidth=1.0
        )
        axes[row_index, column].axvline(0.0, color="0.5", linewidth=0.5)
        axes[row_index, column].set_xlim(-detail_limit, detail_limit)
    synthetic_view = (
        (well["tvdss_m"] >= lower)
        & (well["tvdss_m"] <= upper)
        & np.isfinite(well["synthetic_full"])
        & np.isfinite(reference["synthetic_body"])
    )
    full_x, forward_y = normalized_xy(well["tvdss_m"], well["synthetic_full"], synthetic_view)
    retained_x, retained_y = normalized_xy(well["tvdss_m"], reference["synthetic_body"], synthetic_view)
    axes[row_index, forward_column].plot(full_x, forward_y, color="black", linewidth=1.0)
    axes[row_index, forward_column].plot(retained_x, retained_y, color="#2ca02c", linewidth=1.0)
    for axis in axes[row_index]:
        axis.set_ylim(upper, lower)
    axes[row_index, 0].set_ylabel(f"{well_name}\nTVDSS (m)")
for column, title in enumerate(["real seismic", "full / retained", *list(CANDIDATES), "forward check"]):
    axes[0, column].set_title(title)
candidate_atlas_path = OUTPUT_DIR / "figures" / "body_scale_candidate_atlas.png"
fig.savefig(candidate_atlas_path, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(candidate_atlas_path)))

In [ ]:
# 图 3：尺度与正演贡献汇总。
reference_events = event_scale[np.isclose(event_scale["threshold_fraction"], REFERENCE_EVENT_THRESHOLD)]
fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.5), constrained_layout=True)
for source_name, color in (("real_seismic", "#1f77b4"), ("well_synthetic", "#ff7f0e")):
    values = reference_events.loc[reference_events["source"] == source_name, "width_m"].to_numpy(dtype=np.float64)
    axes[0, 0].hist(values, bins=20, alpha=0.45, color=color, label=source_name)
axes[0, 0].set_xlabel("same-polarity lobe width (m)")
axes[0, 0].legend()
median_tuning = tuning_scales[tuning_scales["frequency_definition"] == "energy_median_hz"]
axes[0, 1].bar(median_tuning["well_name"], median_tuning["tuning_scale_m"], color="#9467bd")
axes[0, 1].tick_params(axis="x", rotation=35)
axes[0, 1].set_ylabel("tuning scale (m)")
for candidate_name, group in candidate_metrics.groupby("candidate"):
    axes[1, 0].scatter(
        [candidate_name] * len(group), group["detail_zero_crossing_p50_m"], alpha=0.75, label=candidate_name
    )
axes[1, 0].tick_params(axis="x", rotation=35)
axes[1, 0].set_ylabel("detail same-sign interval P50 (m)")
for candidate_name, group in candidate_metrics.groupby("candidate"):
    axes[1, 1].scatter(
        group["detail_forward_rms_ratio"], group["body_reconstruction_forward_corr"], alpha=0.8, label=candidate_name
    )
axes[1, 1].set_xlabel("detail forward RMS / full RMS")
axes[1, 1].set_ylabel("corr(full, broad + detail)")
axes[1, 1].legend(fontsize=7)
summary_path = OUTPUT_DIR / "figures" / "scale_summary.png"
fig.savefig(summary_path, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(summary_path)))

## 5. 发布 artifact 与人工评价模板

先看事件尺度图，再看事件窗口候选，最后看汇总指标。形态判断优先于单一自动分数。


In [ ]:
review = candidate_metrics[["well_name", "candidate"]].copy()
for column in (
    "few_continuous_structures",
    "dense_ripples",
    "dog_artifact",
    "stable_across_candidates",
    "interpretable_structure_count",
    "decision",
    "notes",
):
    review[column] = ""
review_path = OUTPUT_DIR / "human_review.csv"
review.to_csv(review_path, index=False)

manifest = {
    "schema": "well_ai_body_scale_decomposition_v1",
    "run_id": RUN_ID,
    "status": "completed",
    "sample_domain": "depth",
    "sample_unit": "m",
    "depth_basis": "tvdss",
    "smoke": SMOKE,
    "source_manifest": str(SOURCE_RUN_DIR / "manifest.json"),
    "seismic_path": str(seismic_path),
    "trusted_wells": list(wells),
    "candidates": CANDIDATES,
    "reference_candidate": REFERENCE_CANDIDATE,
    "event_threshold_fractions": list(EVENT_THRESHOLD_FRACTIONS),
    "reference_event_threshold": REFERENCE_EVENT_THRESHOLD,
    "wavelet_frequency_summary": wavelet_summary,
    "well_artifacts": artifact_paths,
    "well_figures": figure_manifest,
    "tables": {
        "event_scale": str(event_scale_path),
        "tuning_scale": str(OUTPUT_DIR / "tuning_scale.csv"),
        "candidate_metrics": str(candidate_metrics_path),
        "event_candidate_metrics": str(OUTPUT_DIR / "event_candidate_metrics.csv"),
        "sensitivity": str(OUTPUT_DIR / "sensitivity.csv"),
        "human_review": str(review_path),
    },
    "figures": {
        "event_scale_atlas": str(event_atlas_path),
        "candidate_atlas": str(candidate_atlas_path),
        "scale_summary": str(summary_path),
    },
    "interpretation_boundary": "filter decomposition is a scale audit, not geological boundary truth",
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest: {manifest_path}")
print(f"Human review: {review_path}")

## 6. 人工检查顺序

1. `real_event_scale_atlas.png`：确认真实地震事件宽度是否真的接近百米，并比较井上合成是否系统性更窄。
2. `body_scale_candidate_atlas.png`：判断数十米 detail 是少量连续结构，还是规则 DoG 波纹。
3. 逐井 `body_scale_candidate_comparison.png`：确认结论不由单个最漂亮事件造成。
4. `scale_summary.png`：检查视觉结论与厚度、正演贡献是否一致。
5. 在 `human_review.csv` 记录评价。

如果少量连续结构只在单一 FWHM 下出现，本轮应判为负结果。
